# Advanced Problems: Boolean Precedence and Short-Circuiting

Practice problems with full solutions on `and`, `or`, `not`, operator precedence, operand-return behavior, and defensive short-circuit guards.

## Problem 1 — Parenthesize Boolean Expressions

For each expression, predict the value, then verify it. Remember: `not` has higher precedence than `and`, and `and` has higher precedence than `or`.

In [1]:
expressions = {
    "True or False and False": True or False and False,
    "False or True and False": False or True and False,
    "not True or True": not True or True,
    "not (True or True)": not (True or True),
    "True and not False or False": True and not False or False,
    "False or False and not True": False or False and not True,
}

expressions

{'True or False and False': True,
 'False or True and False': False,
 'not True or True': True,
 'not (True or True)': False,
 'True and not False or False': True,
 'False or False and not True': False}

### Solution

`and` binds more tightly than `or`, so `A or B and C` means `A or (B and C)`, not `(A or B) and C`.

## Problem 2 — Show Evaluation Order

Create a function that logs when each condition is evaluated. Use it to prove short-circuiting.

In [2]:
log = []

def check(name, value):
    log.append(name)
    return value

log.clear()
result_1 = check('A', True) or check('B', False)

log_1 = log.copy()

log.clear()
result_2 = check('A', False) and check('B', True)

log_2 = log.copy()

log.clear()
result_3 = check('A', False) or check('B', True) and check('C', False)

result_1, log_1, result_2, log_2, result_3, log

(True, ['A'], False, ['A'], False, ['A', 'B', 'C'])

### Solution

- `or` stops as soon as it finds a truthy operand.
- `and` stops as soon as it finds a falsy operand.
- Because `and` has higher precedence, `A or B and C` evaluates like `A or (B and C)`.

## Problem 3 — Safe Division Guard

Write `is_ratio_greater_than(a, b, threshold)` that avoids division by zero using short-circuiting.

In [3]:
def is_ratio_greater_than(a, b, threshold):
    return bool(b and a / b > threshold)

assert is_ratio_greater_than(10, 2, 2) is True
assert is_ratio_greater_than(10, 5, 2) is False
assert is_ratio_greater_than(10, 0, 2) is False

print('All tests passed.')

All tests passed.


### Best-practice note

This is compact, but it treats every falsy denominator as invalid. For numeric code, an explicit `b != 0` is often clearer.

In [4]:
def is_ratio_greater_than_explicit(a, b, threshold):
    return b != 0 and a / b > threshold

assert is_ratio_greater_than_explicit(10, 0, 2) is False
assert is_ratio_greater_than_explicit(10, 2, 2) is True

print('All tests passed.')

All tests passed.


## Problem 4 — Safe String Validation

Write `starts_with_digit(name)` that returns `True` only when `name` is a non-empty string whose first character is a digit. It must not crash for `None`, empty strings, lists, or integers.

In [5]:
import string

def starts_with_digit(name):
    return isinstance(name, str) and bool(name) and name[0] in string.digits

assert starts_with_digit('1Bob') is True
assert starts_with_digit('Bob') is False
assert starts_with_digit('') is False
assert starts_with_digit(None) is False
assert starts_with_digit(123) is False
assert starts_with_digit(['1Bob']) is False

print('All tests passed.')

All tests passed.


### Solution explanation

The order matters:

1. Check that `name` is a string.
2. Check that it is non-empty.
3. Only then access `name[0]`.

## Problem 5 — `and` and `or` Return Operands

Predict the exact return values, not just their truth values.

In [6]:
results = {
    "0 or 'fallback'": 0 or 'fallback',
    "[] or [1, 2]": [] or [1, 2],
    "'hello' and 99": 'hello' and 99,
    "'' and 99": '' and 99,
    "'x' or expensive": 'x' or print('this will not run'),
    "None or [] or {} or 'done'": None or [] or {} or 'done',
}

results

{"0 or 'fallback'": 'fallback',
 '[] or [1, 2]': [1, 2],
 "'hello' and 99": 99,
 "'' and 99": '',
 "'x' or expensive": 'x',
 "None or [] or {} or 'done'": 'done'}

### Solution

`or` returns the first truthy operand, or the last operand if none are truthy. `and` returns the first falsy operand, or the last operand if all are truthy.

## Problem 6 — Avoid the Default-Value Trap

Fix this broken function:

```python
def page_size(value):
    return value or 20
```

`0` should be allowed because it means "return no records".

In [7]:
def page_size(value):
    if value is None:
        return 20
    if isinstance(value, bool) or not isinstance(value, int):
        raise ValueError('page size must be None or an integer')
    if value < 0:
        raise ValueError('page size cannot be negative')
    return value

assert page_size(None) == 20
assert page_size(0) == 0
assert page_size(5) == 5

for bad in [-1, True, False, '10', 2.5]:
    try:
        page_size(bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f'Expected ValueError for {bad!r}')

print('All tests passed.')

All tests passed.


## Problem 7 — Controlled Lazy Evaluation

Write `first_valid(candidates)` where each candidate is a zero-argument function. Stop as soon as one candidate returns a truthy value.

In [8]:
def first_valid(candidates):
    for candidate in candidates:
        value = candidate()
        if value:
            return value
    return None

calls = []

def c1():
    calls.append('c1')
    return ''

def c2():
    calls.append('c2')
    return []

def c3():
    calls.append('c3')
    return 'valid'

def c4():
    calls.append('c4')
    return 'should not run'

assert first_valid([c1, c2, c3, c4]) == 'valid'
assert calls == ['c1', 'c2', 'c3']

print('All tests passed.')

All tests passed.


## Problem 8 — Validate Usernames Safely

A valid username must be a non-empty string, must not start with a digit, and must contain only ASCII letters, digits, or underscores.

In [9]:
def is_valid_username(username):
    allowed = set(string.ascii_letters + string.digits + '_')
    return (
        isinstance(username, str)
        and bool(username)
        and username[0] not in string.digits
        and all(char in allowed for char in username)
    )

assert is_valid_username('Alice_123') is True
assert is_valid_username('alice') is True
assert is_valid_username('1Alice') is False
assert is_valid_username('') is False
assert is_valid_username(None) is False
assert is_valid_username('Alice Smith') is False
assert is_valid_username('Alice!') is False
assert is_valid_username(123) is False

print('All tests passed.')

All tests passed.


## Problem 9 — Refactor a Risky Conditional

Refactor this unsafe condition:

```python
if record['name'][0] in string.ascii_uppercase and record['active']:
    ...
```

It should not crash if `record` is not a dictionary, keys are missing, or the name is empty.

In [10]:
def should_process(record):
    return (
        isinstance(record, dict)
        and record.get('active') is True
        and isinstance(record.get('name'), str)
        and bool(record.get('name'))
        and record['name'][0] in string.ascii_uppercase
    )

assert should_process({'name': 'Alice', 'active': True}) is True
assert should_process({'name': 'alice', 'active': True}) is False
assert should_process({'name': '', 'active': True}) is False
assert should_process({'active': True}) is False
assert should_process({'name': 'Alice'}) is False
assert should_process(None) is False
assert should_process(['Alice']) is False
assert should_process({'name': 'Alice', 'active': 1}) is False

print('All tests passed.')

All tests passed.


## Problem 10 — Build a Mini Boolean Expression Tracer

Create objects whose truth checks are logged, then use them to inspect evaluation order in complex expressions.

In [11]:
class TracedBool:
    def __init__(self, name, value, log):
        self.name = name
        self.value = value
        self.log = log

    def __bool__(self):
        self.log.append(self.name)
        return self.value

    def __repr__(self):
        return f'TracedBool({self.name!r}, {self.value!r})'

log = []
A = TracedBool('A', False, log)
B = TracedBool('B', True, log)
C = TracedBool('C', False, log)
D = TracedBool('D', True, log)

result = A or B and C or D

print('result:', result)
print('log:', log)

assert result is D
assert log == ['A', 'B', 'C']

print('All tests passed.')

result: TracedBool('D', True)
log: ['A', 'B', 'C']
All tests passed.


### Final Takeaways

- Precedence: `not` > `and` > `or`.
- Use parentheses when readability matters.
- `and` short-circuits on the first falsy operand.
- `or` short-circuits on the first truthy operand.
- `and` and `or` return operands, not always `True` or `False`.
- Put safe checks before risky operations.
- Use `is None` when `None` is semantically different from other falsy values.